# An Automated ER Triage Optimizer

## Overview
This notebook contains the complete PySpark data engineering and machine learning pipeline for the Automated ER Triage Optimizer. The objective of this project is to process complex,relational symptom-to-disease medical data and train a distributed Random Forest classifier to predict patient triage priority (**RED**, **YELLOW**, or **GREEN**).

By automating the initial symptom assessment, this pipeline aims to reduce clinical bottlenecks, eliminate human bias, and ensure critical patients are flagged for immediate intervention.

## Dataset
* **Source:** Symptom-to-Disease Medical Dataset | URL: https://www.kaggle.com/datasets/harrachimustapha/symptom-to-disease-medical-dataset/data
* **Goal:** Map a vectorized array of patient symptoms (`features`) to a priority severity class (`label`).
* **Disclaimer:** This data is strictly synthetic and utilized exclusively for academic algorithmic benchmarking. It is not intended for real-world clinical diagnosis.

**Environment Setup:** Ensure that `pyspark` and `jupyter` are installed in the active Python interpreter before executing the cells below.

In [54]:
!pip install pyspark

In [55]:
import pyspark

In [56]:
print(f"PySpark Wrapper Version: {pyspark.__version__}")

PySpark Wrapper Version: 3.5.0


### Initialize the Spark Session

In [57]:
from pyspark.sql import SparkSession

In [58]:
# Initialize the distributed Spark Session
spark = SparkSession.builder \
    .appName("ERTriageOptimizer") \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

print("Spark Session successfully created!")

Spark Session successfully created!


In [59]:
print(f"Apache Spark Engine Version: {spark.version}")

Apache Spark Engine Version: 3.5.0


### Data Ingestion

In [60]:
# Load CSV files into PySpark DataFrames from Kaggle's input directory
patients_df = spark.read.csv("datasets/patients.csv", header=True, inferSchema=True)
patient_evidences_df = spark.read.csv("datasets/patient_evidences.csv", header=True, inferSchema=True)
diseases_df = spark.read.csv("datasets/diseases.csv", header=True, inferSchema=True)

In [61]:
patients_df.show(5)

+----------+---+---+-----------------------+-------------------------+--------------------+-------------------+-------------------------+-------------+--------------------------+-----+
|patient_id|age|sex|ground_truth_disease_id|ground_truth_disease_name|initial_evidence_raw|initial_evidence_id|initial_evidence_value_id|num_evidences|num_differential_diagnoses|split|
+----------+---+---+-----------------------+-------------------------+--------------------+-------------------+-------------------------+-------------+--------------------------+-----+
|  P0000001| 18|  M|                   D038|                     URTI|                E_91|               E_91|                     NULL|           19|                         8|train|
|  P0000002| 21|  M|                   D006|     HIV (initial infe...|                E_50|               E_50|                     NULL|           31|                         4|train|
|  P0000003| 19|  F|                   D040|                Pneumonia|     

In [62]:
patient_evidences_df.show(5)

+----------+------------+-----------+-----------------+-------------+---------+-------------------+-----+
|patient_id|evidence_raw|evidence_id|evidence_value_id|evidence_type|data_type|is_initial_evidence|split|
+----------+------------+-----------+-----------------+-------------+---------+-------------------+-----+
|  P0000001|        E_48|       E_48|             NULL|   antecedent|        B|              false|train|
|  P0000001|        E_50|       E_50|             NULL|      symptom|        B|              false|train|
|  P0000001|        E_53|       E_53|             NULL|      symptom|        B|              false|train|
|  P0000001|E_54_@_V_161|       E_54|            V_161|      symptom|        M|              false|train|
|  P0000001|E_54_@_V_183|       E_54|            V_183|      symptom|        M|              false|train|
+----------+------------+-----------+-----------------+-------------+---------+-------------------+-----+
only showing top 5 rows



In [63]:
diseases_df.show(5)

+----------+--------------------+--------------------+--------+--------+--------------------+-----------------------+
|disease_id|        disease_name|     disease_name_en|icd10_id|severity|num_defined_symptoms|num_defined_antecedents|
+----------+--------------------+--------------------+--------+--------+--------------------+-----------------------+
|      D001|Spontaneous pneum...|Spontaneous pneum...|     J93|       2|                  12|                      5|
|      D002|    Cluster headache|    Cluster headache| g44.009|       3|                  10|                      4|
|      D003|           Boerhaave|           Boerhaave|   K22.3|       2|                  11|                      2|
|      D004|Spontaneous rib f...|Spontaneous rib f...|   S22.9|       3|                  13|                      4|
|      D005|                GERD|                GERD|     K21|       3|                  13|                      7|
+----------+--------------------+--------------------+--

### Data Cleaning

In [64]:
# Prune Patients DataFrame
patients_df = patients_df.drop(
    "ground_truth_disease_name",
    "num_differential_diagnoses",
    "initial_evidence_raw",
    "split"
)

In [65]:
patients_df.show(5)

+----------+---+---+-----------------------+-------------------+-------------------------+-------------+
|patient_id|age|sex|ground_truth_disease_id|initial_evidence_id|initial_evidence_value_id|num_evidences|
+----------+---+---+-----------------------+-------------------+-------------------------+-------------+
|  P0000001| 18|  M|                   D038|               E_91|                     NULL|           19|
|  P0000002| 21|  M|                   D006|               E_50|                     NULL|           31|
|  P0000003| 19|  F|                   D040|               E_77|                     NULL|           34|
|  P0000004| 34|  F|                   D038|               E_53|                     NULL|           16|
|  P0000005| 36|  M|                   D038|              E_201|                     NULL|           17|
+----------+---+---+-----------------------+-------------------+-------------------------+-------------+
only showing top 5 rows



In [66]:
# Prune Patient Evidences DataFrame
patient_evidences_df = patient_evidences_df.drop("evidence_raw", "split")

In [67]:
patient_evidences_df.show(5)

+----------+-----------+-----------------+-------------+---------+-------------------+
|patient_id|evidence_id|evidence_value_id|evidence_type|data_type|is_initial_evidence|
+----------+-----------+-----------------+-------------+---------+-------------------+
|  P0000001|       E_48|             NULL|   antecedent|        B|              false|
|  P0000001|       E_50|             NULL|      symptom|        B|              false|
|  P0000001|       E_53|             NULL|      symptom|        B|              false|
|  P0000001|       E_54|            V_161|      symptom|        M|              false|
|  P0000001|       E_54|            V_183|      symptom|        M|              false|
+----------+-----------+-----------------+-------------+---------+-------------------+
only showing top 5 rows



In [68]:
# Prune Diseases DataFrame
diseases_df = diseases_df.drop(
    "disease_name",
    "disease_name_en",
    "icd10_id",
    "num_defined_symptoms",
    "num_defined_antecedents"
)

In [69]:
diseases_df.show(5)

+----------+--------+
|disease_id|severity|
+----------+--------+
|      D001|       2|
|      D002|       3|
|      D003|       2|
|      D004|       3|
|      D005|       3|
+----------+--------+
only showing top 5 rows



### Distributed Joins

In [70]:
from pyspark.sql.functions import collect_list

In [71]:
# Aggregate multiple symptoms into a single array per patient
patients_evidences_agg_df = patient_evidences_df.groupBy("patient_id") \
    .agg(collect_list("evidence_id").alias("symptoms_array"))

In [72]:
patients_evidences_agg_df.show(5)

+----------+--------------------+
|patient_id|      symptoms_array|
+----------+--------------------+
|  P0000003|[E_53, E_54, E_54...|
|  P0000017|[E_25, E_53, E_54...|
|  P0000025|[E_45, E_78, E_16...|
|  P0000029|[E_53, E_54, E_54...|
|  P0000030|[E_6, E_51, E_53,...|
+----------+--------------------+
only showing top 5 rows



In [73]:
# Join patients with their aggregated symptoms
patients_with_symptoms_df = patients_df.join(
    patients_evidences_agg_df, on="patient_id", how="inner"
)

In [74]:
patients_with_symptoms_df.show(5)

+----------+---+---+-----------------------+-------------------+-------------------------+-------------+--------------------+
|patient_id|age|sex|ground_truth_disease_id|initial_evidence_id|initial_evidence_value_id|num_evidences|      symptoms_array|
+----------+---+---+-----------------------+-------------------+-------------------------+-------------+--------------------+
|  P0000003| 19|  F|                   D040|               E_77|                     NULL|           34|[E_53, E_54, E_54...|
|  P0000017| 57|  F|                   D002|               E_53|                     NULL|           21|[E_25, E_53, E_54...|
|  P0000025| 11|  M|                   D028|               E_45|                     NULL|            6|[E_45, E_78, E_16...|
|  P0000029| 15|  F|                   D003|               E_53|                     NULL|           21|[E_53, E_54, E_54...|
|  P0000030| 59|  M|                   D047|              E_162|                     NULL|           27|[E_6, E_51, E_

In [75]:
# Join with diseases to append the severity score, then drop temporary IDs
master_df = patients_with_symptoms_df.join(
    diseases_df,
    patients_with_symptoms_df.ground_truth_disease_id == diseases_df.disease_id,
    how="inner"
).drop("patient_id", "ground_truth_disease_id", "disease_id")

In [76]:
master_df.show(5)

+---+---+-------------------+-------------------------+-------------+--------------------+--------+
|age|sex|initial_evidence_id|initial_evidence_value_id|num_evidences|      symptoms_array|severity|
+---+---+-------------------+-------------------------+-------------+--------------------+--------+
| 19|  F|               E_77|                     NULL|           34|[E_53, E_54, E_54...|       3|
| 57|  F|               E_53|                     NULL|           21|[E_25, E_53, E_54...|       3|
| 11|  M|               E_45|                     NULL|            6|[E_45, E_78, E_16...|       3|
| 15|  F|               E_53|                     NULL|           21|[E_53, E_54, E_54...|       2|
| 59|  M|              E_162|                     NULL|           27|[E_6, E_51, E_53,...|       3|
+---+---+-------------------+-------------------------+-------------+--------------------+--------+
only showing top 5 rows



### Data Transformations

In [77]:
from pyspark.sql.functions import col, when
from pyspark.ml.feature import CountVectorizer, StringIndexer

In [78]:
# Categorize severity into clinical RED, YELLOW, and GREEN classes
master_df = master_df.withColumn(
    "triage_priority",
    when(col("severity") >= 4, "RED")
    .when(col("severity") == 3, "YELLOW")
    .otherwise("GREEN")
)

### Export Master Dataset

In [79]:
#Convert the PySpark DataFrame to a standard Pandas DataFrame
master_pandas_df = master_df.toPandas()

In [80]:
# Save it to Kaggle's writable "working" directory
master_pandas_df.to_csv("datasets/master_dataset.csv", index=False)

### Further Transformations

In [81]:
# Convert symptom arrays into dense matrices
vectorizer = CountVectorizer(inputCol="symptoms_array", outputCol="features")

In [82]:
# Encode categorical target labels to numerical indices
indexer = StringIndexer(inputCol="triage_priority", outputCol="label")

### Master Dataset Analysis

In [83]:
from pyspark.sql.functions import round

In [84]:
#Check total number of records
total_rows = master_df.count()
print(f"Total number of records: {total_rows}\n")

Total number of records: 1292579



In [85]:
# Group by triage_priority and calculate counts
balance_df = master_df.groupBy("triage_priority").count()

In [86]:
# Add a percentage column for easier reading
balance_df = balance_df.withColumn(
    "Percentage (%)", 
    round((col("count") / total_rows) * 100, 2)
)

In [87]:
balance_df.show()

+---------------+------+--------------+
|triage_priority| count|Percentage (%)|
+---------------+------+--------------+
|            RED|540859|         41.84|
|          GREEN|362316|         28.03|
|         YELLOW|389404|         30.13|
+---------------+------+--------------+



### The Machine Learning Pipeline

In [88]:
# Split data into Training (80%) and Testing (20%)
train_data, test_data = master_df.randomSplit([0.8, 0.2], seed=42)

In [89]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml import Pipeline

In [90]:
# Initialize the three Evaluators (Accuracy, Precision, F1-Score)
print("Initializing evaluators...")
acc_eval = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")
prec_eval = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="weightedPrecision")
f1_eval = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="f1")
print("Complete")

Initializing evaluators...
Complete


In [91]:
from pyspark.ml.classification import RandomForestClassifier

In [92]:
# Initialize Random Forest Classifier
rf_classifier = RandomForestClassifier(
    featuresCol="features",
    labelCol="label",
    numTrees=50
)

# Encapsulate all stages into a unified Pipeline
triage_pipeline = Pipeline(stages=[vectorizer, indexer, rf_classifier])

In [93]:
# Train the model
print("Training the model...")
rf_model = triage_pipeline.fit(train_data)
print("Complete")

Training the model...
Complete


In [94]:
# Generate the predictions
print("Generating predictions...")
rf_pred = rf_model.transform(test_data)
print("Complete")

Generating predictions...
Complete


In [95]:
# Evaluate the already trained Random Forest Classifier
rf_acc = acc_eval.evaluate(rf_pred)
rf_prec = prec_eval.evaluate(rf_pred)
rf_f1 = f1_eval.evaluate(rf_pred)

In [96]:
print("Random Forest Classifier")
print(f"Accuracy: {rf_acc}")
print(f"Precision: {rf_prec}")
print(f"F1 Score: {rf_f1}")

Random Forest Classifier
Accuracy: 0.8324378855579091
Precision: 0.8527638963954824
F1 Score: 0.8314437634461967


### Introducing Principal Component Analysis

In [97]:
from pyspark.ml.feature import PCA

In [98]:
# Compress the 'features' matrix into 50 principal components
# If you get an error saying 'k is larger than the number of features', lower k to 20 or 30
print("Initializing...")
pca = PCA(k=50, inputCol="features", outputCol="compressed_features")
print("Complete")

Initializing...
Complete


In [99]:
# Initialize PCA-Optimized Random Forest Classifier
rf_classifier = RandomForestClassifier(
    featuresCol="compressed_features",
    labelCol="label",
    numTrees=50
)

# Encapsulate all stages into a unified Pipeline
pca_pipeline = Pipeline(stages=[vectorizer, pca, indexer, rf_classifier])

In [100]:
# Train the model
print("Training the model...")
pca_model = pca_pipeline.fit(train_data)
print("Complete")

Training the model...
Complete


In [101]:
# Generate the predictions
print("Generating predictions...")
pca_pred = pca_model.transform(test_data)
print("Complete")

Generating predictions...
Complete


In [102]:
# Evaluate the PCA-Optimized Random Forest Classifier
pca_acc = acc_eval.evaluate(pca_pred)
pca_prec = prec_eval.evaluate(pca_pred)
pca_f1 = f1_eval.evaluate(pca_pred)

In [103]:
print(f"PCA-Optimized Random Forest:")
print(f"Accuracy:  {pca_acc}")
print(f"Precision: {pca_prec}")
print(f"F1-Score:  {pca_f1}")

PCA-Optimized Random Forest:
Accuracy:  0.9258453284682818
Precision: 0.9307344476224564
F1-Score:  0.925175315841514


### Train Logistic Regression

In [104]:
from pyspark.ml.classification import LogisticRegression

In [105]:
# Train the model
print("Training the model...")
lr = LogisticRegression(featuresCol="features", labelCol="label", maxIter=10)
lr_pipeline = Pipeline(stages=[vectorizer, indexer, lr])
lr_model = lr_pipeline.fit(train_data)
print("Complete")

Training the model...
Complete


In [106]:
# Generate the predictions
print("Generating predictions...")
lr_preds = lr_model.transform(test_data)
print("Complete")

Generating predictions...
Complete


In [107]:
# Evaluate the baseline model
lr_acc = acc_eval.evaluate(lr_preds)
lr_prec = prec_eval.evaluate(lr_preds)
lr_f1 = f1_eval.evaluate(lr_preds)

In [108]:
print("Logistic Regression")
print(f"Accuracy: {lr_acc}")
print(f"Precision: {lr_prec}")
print(f"F1 Score: {lr_f1}")

Logistic Regression
Accuracy: 0.9983650025510599
Precision: 0.9983667014510154
F1 Score: 0.9983649989495691


### Train Decision Tree Classifier

In [109]:
from pyspark.ml.classification import DecisionTreeClassifier

In [110]:
# Train the model
print("Training the model...")
dt = DecisionTreeClassifier(featuresCol="features", labelCol="label")
dt_pipeline = Pipeline(stages=[vectorizer, indexer, dt])
dt_model = dt_pipeline.fit(train_data)
print("Complete")

Training the model...
Complete


In [111]:
# Generate the predictions
print("Generating predictions...")
dt_preds = dt_model.transform(test_data)
print("Complete")

Generating predictions...
Complete


In [112]:
# Evaluate the Decision Tree Classifier
dt_acc = acc_eval.evaluate(dt_preds)
dt_prec = prec_eval.evaluate(dt_preds)
dt_f1 = f1_eval.evaluate(dt_preds)

In [113]:
print("Decision Tree Classifier")
print(f"Accuracy: {dt_acc}")
print(f"Precision: {dt_prec}")
print(f"F1 Score: {dt_f1}")

Decision Tree Classifier
Accuracy: 0.8322484886903013
Precision: 0.8385765468635524
F1 Score: 0.8286996863578231


### PCA-Optimized Decision Tree Classifier

In [114]:
# Train the model
print("Training the model...")
pca_dt = DecisionTreeClassifier(featuresCol="compressed_features", labelCol="label")
pca_dt_pipeline = Pipeline(stages=[vectorizer, pca, indexer, pca_dt])
pca_dt_model = pca_dt_pipeline.fit(train_data)
print("Complete")

Training the model...
Complete


In [115]:
# Generate the predictions
print("Generating predictions...")
pca_dt_preds = pca_dt_model.transform(test_data)
print("Complete")

Generating predictions...
Complete


In [116]:
# Evaluate PCA-Optimized Decision Tree Classifier
pca_dt_acc = acc_eval.evaluate(pca_dt_preds)
pca_dt_prec = prec_eval.evaluate(pca_dt_preds)
pca_dt_f1 = f1_eval.evaluate(pca_dt_preds)

In [117]:
print("PCA-Optimized Decision Tree Classifier")
print(f"Accuracy: {pca_dt_acc}")
print(f"Precision: {pca_dt_prec}")
print(f"F1 Score: {pca_dt_f1}")

PCA-Optimized Decision Tree Classifier
Accuracy: 0.8689257718888665
Precision: 0.880539255421112
F1 Score: 0.8680770529816122


### PCA-Optimized Logistic Regression

In [118]:
# Train the model
print("Training the model...")
pca_lr = LogisticRegression(featuresCol="compressed_features", labelCol="label", maxIter=10)
pca_lr_pipeline = Pipeline(stages=[vectorizer, pca, indexer, pca_lr])
pca_lr_model = pca_lr_pipeline.fit(train_data)
print("Complete")

Training the model...
Complete


In [119]:
# Generate the predictions
print("Generating predictions...")
pca_lr_preds = pca_lr_model.transform(test_data)
print("Complete")

Generating predictions...
Complete


In [120]:
# Evaluate the PCA-Optimised baseline model
pca_lr_acc = acc_eval.evaluate(lr_preds)
pca_lr_prec = prec_eval.evaluate(lr_preds)
pca_lr_f1 = f1_eval.evaluate(lr_preds)

In [121]:
print("PCA-Optimized Logistic Regression")
print(f"Accuracy: {pca_lr_acc}")
print(f"Precision: {pca_lr_prec}")
print(f"F1 Score: {pca_lr_f1}")

PCA-Optimized Logistic Regression
Accuracy: 0.9983650025510599
Precision: 0.9983667014510154
F1 Score: 0.9983649989495691


### Performance Analysis

In [122]:
print(f"{'Model':<32} | {'Accuracy':<9} | {'Precision':<9} | {'F1-Score':<9}")
print("-" * 68)
print(f"{'Logistic Regression (Baseline)':<32} | {lr_acc:.3f}     | {lr_prec:.3f}     | {lr_f1:.3f}")
print(f"{'Decision Tree Classifier':<32} | {dt_acc:.3f}     | {dt_prec:.3f}     | {dt_f1:.3f}")
print(f"{'Random Forest':<32} | {rf_acc:.3f}     | {rf_prec:.3f}     | {rf_f1:.3f}")
print(f"{'PCA-Optimized Logistic Regression':<32} | {pca_lr_acc:.3f}     | {pca_lr_prec:.3f}     | {pca_lr_f1:.3f}")
print(f"{'PCA-Optimized Decision Tree':<32} | {pca_dt_acc:.3f}     | {pca_dt_prec:.3f}     | {pca_dt_f1:.3f}")
print(f"{'PCA-Optimized Random Forest':<32} | {pca_acc:.3f}     | {pca_prec:.3f}     | {pca_f1:.3f}")


Model                            | Accuracy  | Precision | F1-Score 
--------------------------------------------------------------------
Logistic Regression (Baseline)   | 0.998     | 0.998     | 0.998
Decision Tree Classifier         | 0.832     | 0.839     | 0.829
Random Forest                    | 0.832     | 0.853     | 0.831
PCA-Optimized Logistic Regression | 0.998     | 0.998     | 0.998
PCA-Optimized Decision Tree      | 0.869     | 0.881     | 0.868
PCA-Optimized Random Forest      | 0.926     | 0.931     | 0.925
